In [1]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Dense,
    Flatten,
    Concatenate,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model

In [2]:
route_dataset_df = pd.read_pickle(
    "../data/processed/route_dataset.pkl"
)

print(route_dataset_df.shape)

(19647, 15)


In [3]:
route_dataset_df["Label"] = (
    route_dataset_df["Label"]
    .map({
        "ND": 0,
        "D": 1
    })
    .astype(np.int32)
)

print(
    route_dataset_df["Label"].value_counts()
)

Label
1    12158
0     7489
Name: count, dtype: int64


In [4]:
MAX_ROUTE_LENGTH = route_dataset_df[
    "Planned Route"
].apply(len).max()

print(MAX_ROUTE_LENGTH)

36


In [5]:
route_sequences = pad_sequences(
    route_dataset_df["Planned Route"],
    maxlen=MAX_ROUTE_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

print(route_sequences.shape)

(19647, 36)


In [6]:
max_stop_id = max([
    max(route)
    for route in route_dataset_df["Planned Route"]
])

VOCAB_SIZE = max_stop_id + 1

print(VOCAB_SIZE)

10864


In [7]:
driver_input = route_dataset_df[
    ["Driver ID"]
].values

numerical_features = route_dataset_df[
    [
        "Country",
        "Week ID",
        "Planned Stop Count",
        "Planned Distance"
    ]
].values

y = route_dataset_df[
    "Label"
].values


In [8]:
driver_train, driver_test, \
route_train, route_test, \
num_train, num_test, \
y_train, y_test = train_test_split(
    driver_input,
    route_sequences,
    numerical_features,
    y,
    test_size=0.2,
    random_state=42
)

print(driver_train.shape)
print(route_train.shape)
print(num_train.shape)
print(y_train.shape)

(15717, 1)
(15717, 36)
(15717, 4)
(15717,)


In [9]:
driver_layer = Input(
    shape=(1,),
    name="Driver_Input"
)

driver_embedding = Embedding(
    input_dim=5000,
    output_dim=32
)(driver_layer)

driver_embedding = Flatten()(
    driver_embedding
)

route_layer = Input(
    shape=(MAX_ROUTE_LENGTH,),
    name="Route_Input"
)

route_embedding = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=64
)(
    route_layer
)

attention_output = MultiHeadAttention(
    num_heads=4,
    key_dim=16
)(
    route_embedding,
    route_embedding
)

attention_output = LayerNormalization()(
    attention_output
)

route_features = GlobalAveragePooling1D()(
    attention_output
)

numerical_layer = Input(
    shape=(4,),
    name="Numerical_Input"
)

merged = Concatenate()(
    [
        driver_embedding,
        route_features,
        numerical_layer
    ]
)

dense = Dense(
    64,
    activation="relu"
)(
    merged
)

dense = Dense(
    32,
    activation="relu"
)(
    dense
)

output = Dense(
    1,
    activation="sigmoid"
)(
    dense
)

model = Model(
    inputs=[
        driver_layer,
        route_layer,
        numerical_layer
    ],
    outputs=output
)

In [11]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Route_Input         │ (None, 36)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 36, 64)    │    695,296 │ Route_Input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Driver_Input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 36, 64)    │     16,640 │ embedding_3[0][0… │
│ (MultiHeadAttentio… │                   │            │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 32)     │    160,000 │ Driver_Input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 36, 64)    │        128 │ multi_head_atten… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Numerical_Input     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 100)       │          0 │ flatten_1[0][0],  │
│ (Concatenate)       │                   │            │ global_average_p… │
│                     │                   │            │ Numerical_Input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      6,464 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      2,080 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         33 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 880,641 (3.36 MB)

 Trainable params: 880,641 (3.36 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(
    [
        driver_train,
        route_train,
        num_train
    ],
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=512,
    verbose=1
)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - accuracy: 0.6863 - loss: 0.5968 - val_accuracy: 0.7236 - val_loss: 0.5552
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.7520 - loss: 0.5112 - val_accuracy: 0.7392 - val_loss: 0.5142
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.7847 - loss: 0.4534 - val_accuracy: 0.7529 - val_loss: 0.4817
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - accuracy: 0.8098 - loss: 0.4067 - val_accuracy: 0.7287 - val_loss: 0.5567
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.8224 - loss: 0.3848 - val_accuracy: 0.7627 - val_loss: 0.4970
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.8444 - loss: 0.3426 - val_accuracy: 0.7344 - val_loss: 0.5812
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.8607 - loss: 0.3156 - val_accuracy: 0.7440 - val_loss: 0.5898
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.8795 - loss: 0.2803 - val_accuracy: 0.7478 - v

In [13]:
y_prob = model.predict(
    [
        driver_test,
        route_test,
        num_test
    ]
)

y_pred = (
    y_prob > 0.5
).astype(int)

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [14]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("=" * 60)
print("ATTENTION CLASSIFICATION RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")

print()

print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    "Confusion Matrix:\n"
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

ATTENTION CLASSIFICATION RESULTS
Accuracy : 0.7265

              precision    recall  f1-score   support

           0       0.65      0.62      0.63      1500
           1       0.77      0.80      0.78      2430

    accuracy                           0.73      3930
   macro avg       0.71      0.71      0.71      3930
weighted avg       0.72      0.73      0.72      3930

Confusion Matrix:

[[ 923  577]
 [ 498 1932]]


In [15]:
model.save(
    "../outputs/models/attention_classification.keras"
)

print(
    "Attention Classification model saved successfully!"
)

Attention Classification model saved successfully!
